In [0]:
%pip install -q hdbscan umap-learn scikit-learn pandas plotly
dbutils.library.restartPython()


In [0]:
from pyspark.sql import functions as F

TABLE = "noc_documents_azure"

df = (spark.table(TABLE)
      .select("path", "is_scanned", "final_text", "embedding")
      .where(F.col("embedding").isNotNull()))

df = df.withColumn("text_preview", F.substring(F.col("final_text"), 1, 300))

df = df.drop("final_text")

print("Rows:", df.count())


In [0]:
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.feature import PCA

to_vec = F.udf(lambda x: Vectors.dense(x), VectorUDT())
df2 = df.withColumn("features", to_vec("embedding")).drop("embedding")

PCA_K = 50
pca = PCA(k=PCA_K, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(df2)

df_pca = (pca_model.transform(df2)
          .select("path", "is_scanned", "text_preview", "pca_features"))


In [0]:
import numpy as np

pdf = df_pca.toPandas()

X = np.vstack(
    pdf["pca_features"].apply(lambda v: np.array(v.toArray(), dtype=np.float32)).values
)

norms = np.linalg.norm(X, axis=1, keepdims=True)
norms[norms == 0] = 1.0
Xn = X / norms

print("Collected rows:", len(pdf))
print("Xn shape:", Xn.shape, "dtype:", Xn.dtype)


In [0]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30, 
    min_samples=5,
    metric="euclidean"
)

labels = clusterer.fit_predict(Xn)
pdf["cluster"] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct = float(np.mean(labels == -1)) * 100

print("Clusters:", n_clusters)
print("Noise %:", round(noise_pct, 2))


In [0]:
import umap

umap_2d = umap.UMAP(
    n_neighbors=30,
    min_dist=0.05,
    n_components=2,
    metric="euclidean",
    random_state=42
)

X2 = umap_2d.fit_transform(Xn)
pdf["umap_x"] = X2[:, 0]
pdf["umap_y"] = X2[:, 1]


In [0]:
import plotly.express as px

fig = px.scatter(
    pdf,
    x="umap_x",
    y="umap_y",
    color="cluster",
    hover_data={
        "path": True,
        "is_scanned": True,
        "text_preview": True,
        "cluster": True,
        "umap_x": False,
        "umap_y": False
    },
    title=f"Document Clusters (HDBSCAN) — clusters={n_clusters}, noise={noise_pct:.1f}%",
    height=750
)

fig.update_traces(marker=dict(size=5, opacity=0.8))
fig.show()


In [0]:
out_pdf = pdf[["path", "is_scanned", "cluster"]].copy()
out_sdf = spark.createDataFrame(out_pdf)

OUT_TABLE = "noc_documents_azure_hdbscan_clusters_full"

(out_sdf.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(OUT_TABLE))

print("Saved:", OUT_TABLE)


In [0]:
plot_pdf = pdf.sample(n=15000, random_state=42)


In [0]:
fig.show()

In [0]:
plot_pdf = pdf.sample(n=12000, random_state=42)

fig = px.scatter(
    plot_pdf,
    x="umap_x",
    y="umap_y",
    color="cluster",
    hover_data={
        "path": True,
        "cluster": True,
        "text_preview": True
    },
    height=750
)

fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()


In [0]:
pdf["cluster"].value_counts().head(20)


In [0]:
PCA_K = 100


In [0]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=80,
    min_samples=20,
    metric="euclidean"
)


In [0]:
big = pdf["cluster"].value_counts().idxmax()

sub_pdf = pdf[pdf["cluster"] == big].copy()
X_sub = Xn[pdf["cluster"].values == big]   

sub_clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric="euclidean"
)

sub_labels = sub_clusterer.fit_predict(X_sub)
sub_pdf["sub_cluster"] = sub_labels

sub_pdf["sub_cluster"].value_counts().head(20)


In [0]:
import plotly.express as px

fig = px.scatter(pdf, x="umap_x", y="umap_y", color="is_scanned",
                 hover_data=["path", "text_preview"])
fig.show()


In [0]:
import plotly.express as px

plot_pdf = pdf.sample(n=12000, random_state=42)  

fig = px.scatter(
    plot_pdf,
    x="umap_x", y="umap_y",
    color="is_scanned",
    hover_data=["path", "text_preview"],
    height=750
)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()


In [0]:
import plotly.express as px

plot_pdf = pdf.sample(n=12000, random_state=42)  # or full if exporting

fig = px.scatter(
    plot_pdf,
    x="umap_x",
    y="umap_y",
    color="cluster",
    height=750
)

# Build clean hover text
plot_pdf["hover_label"] = (
    "<b>File:</b> " + plot_pdf["path"].astype(str) +
    "<br><b>Cluster:</b> " + plot_pdf["cluster"].astype(str) +
    "<br><br><b>Preview:</b><br>" +
    plot_pdf["text_preview"].astype(str)
)

fig.update_traces(
    customdata=plot_pdf[["hover_label"]],
    hovertemplate="%{customdata[0]}<extra></extra>",
    marker=dict(size=6, opacity=0.8)
)

fig.show()


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.feature import PCA

TABLE = "noc_documents_azure"

df = (spark.table(TABLE)
      .select("path", "is_scanned", "embedding")
      .where(F.col("embedding").isNotNull()))

# Convert array -> Spark vector
to_vec = F.udf(lambda x: Vectors.dense(x), VectorUDT())
df = df.withColumn("features", to_vec("embedding"))

PCA_K = 100  # increase for better separation
pca = PCA(k=PCA_K, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(df)

df_pca = pca_model.transform(df).select("path", "is_scanned", "pca_features")

print("Rows:", df_pca.count())


In [0]:
import numpy as np

pdf = df_pca.toPandas()

Xp = np.vstack(
    pdf["pca_features"].apply(lambda v: np.array(v.toArray(), dtype=np.float32)).values
)

print("Xp shape:", Xp.shape)


In [0]:
from sklearn.cluster import KMeans

K = 6

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = kmeans.fit_predict(Xp)

pdf["cluster"] = labels

print("Clusters:", len(set(labels)))
print(pdf["cluster"].value_counts())


In [0]:
import umap

umap_2d = umap.UMAP(
    n_neighbors=40,
    min_dist=0.25,
    n_components=2,
    metric="euclidean",
    random_state=42
)

X2 = umap_2d.fit_transform(Xp)

pdf["umap_x"] = X2[:, 0]
pdf["umap_y"] = X2[:, 1]


In [0]:
import plotly.express as px
import os

pdf["file_name"] = pdf["path"].apply(lambda x: os.path.basename(str(x)))

plot_pdf = pdf.sample(n=15000, random_state=42)

fig = px.scatter(
    plot_pdf,
    x="umap_x",
    y="umap_y",
    color="cluster",
    height=800
)

fig.update_traces(
    customdata=plot_pdf[["file_name"]],
    hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
    marker=dict(size=6, opacity=0.85)
)

fig.update_layout(
    title="Document Clusters (KMeans + UMAP)",
    template="plotly_dark"
)

fig.show()


In [0]:
import umap
import numpy as np

umap_3d = umap.UMAP(
    n_neighbors=40,
    min_dist=0.25,
    n_components=3,
    metric="euclidean",
    random_state=42
)

X3 = umap_3d.fit_transform(Xp)

pdf["umap_x"] = X3[:, 0]
pdf["umap_y"] = X3[:, 1]
pdf["umap_z"] = X3[:, 2]


In [0]:
import plotly.express as px
import os

pdf["file_name"] = pdf["path"].apply(lambda x: os.path.basename(str(x)))

plot_pdf = pdf.sample(n=12000, random_state=42)  # keep it manageable

fig = px.scatter_3d(
    plot_pdf,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="cluster",
    height=800
)

fig.update_traces(
    customdata=plot_pdf[["file_name"]],
    hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
    marker=dict(size=3, opacity=0.85)
)

fig.update_layout(
    title="3D Document Clusters (KMeans + 3D UMAP)",
    template="plotly_dark"
)

fig.show()



In [0]:
out_path = "/dbfs/FileStore/umap_3d_clusters.html"
fig.write_html(out_path)
print("Open in browser at: /files/umap_3d_clusters.html")


In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

# ---- Topic extraction function ----
def extract_cluster_topics(texts, n=12):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1,2),   # words + phrases
        max_features=10000,
        min_df=5
    )
    
    X = vectorizer.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    
    top = scores.argsort()[::-1][:n]
    return [terms[i] for i in top]


# ---- Generate topics per KMeans cluster ----
cluster_topics = {}

for cid in sorted(pdf_full["cluster"].unique()):
    texts = pdf_full[pdf_full["cluster"] == cid]["final_text"].dropna()
    
    if len(texts) > 20:
        cluster_topics[cid] = extract_cluster_topics(texts, 15)

cluster_topics


In [0]:
summary = []

for cid, words in cluster_topics.items():
    summary.append({
        "cluster": cid,
        "size": len(pdf_full[pdf_full["cluster"] == cid]),
        "top_keywords": ", ".join(words[:10])
    })

pd.DataFrame(summary).sort_values("size", ascending=False)


In [0]:
print("Min/Max cluster:", pdf["cluster"].min(), pdf["cluster"].max())


In [0]:
# ============================================================
# FULL PIPELINE (ONE BLOCK): KMeans (K=7) + UMAP 2D + Plotly
# - Reads noc_documents_azure (path, is_scanned, embedding)
# - PCA in Spark (distributed) -> 100 dims (safe)
# - KMeans in sklearn -> K clusters (you control K)
# - UMAP 2D for visualization
# - Stratified sample for plotting so ALL clusters appear
# - Hover shows ONLY file name
# - (Optional) Save cluster assignments to Delta table
# ============================================================

# ---------- installs (run once) ----------
# %pip install -q pandas numpy plotly umap-learn scikit-learn
# dbutils.library.restartPython()

from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.feature import PCA as SparkPCA

import numpy as np
import pandas as pd
import os

from sklearn.cluster import KMeans
import umap
import plotly.express as px

# -----------------------------
# CONFIG
# -----------------------------
TABLE = "noc_documents_azure"
K = 25                    # <<< number of clusters you want
PCA_K = 100              # keep more structure for better separation
UMAP_NEIGHBORS = 40
UMAP_MIN_DIST = 0.25
PLOT_PER_CLUSTER = 2000  # stratified sample size per cluster for plot
SEED = 42

SAVE_RESULTS = True
OUT_TABLE = "noc_documents_azure_kmeans_clusters_k7"

# -----------------------------
# 1) LOAD (READ-ONLY)
# -----------------------------
df = (spark.table(TABLE)
      .select("path", "is_scanned", "embedding")
      .where(F.col("embedding").isNotNull()))

# -----------------------------
# 2) SPARK PCA (DISTRIBUTED)
# -----------------------------
to_vec = F.udf(lambda x: Vectors.dense(x), VectorUDT())
df = df.withColumn("features", to_vec("embedding")).drop("embedding")

pca = SparkPCA(k=PCA_K, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(df)

df_pca = pca_model.transform(df).select("path", "is_scanned", "pca_features")
print("Rows:", df_pca.count())

# -----------------------------
# 3) COLLECT PCA FEATURES (SAFE)
# -----------------------------
pdf = df_pca.toPandas()

Xp = np.vstack(
    pdf["pca_features"].apply(lambda v: np.array(v.toArray(), dtype=np.float32)).values
)
print("Xp shape:", Xp.shape, "dtype:", Xp.dtype)

# -----------------------------
# 4) KMEANS CLUSTERING (K CLUSTERS)
# -----------------------------
kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10)
labels = kmeans.fit_predict(Xp)
pdf["cluster"] = labels

print("\nKMeans clusters present:", sorted(pdf["cluster"].unique()))
print("Cluster counts:\n", pdf["cluster"].value_counts().sort_index())

# -----------------------------
# 5) UMAP 2D (FOR VISUAL)
# -----------------------------
umap_2d = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    n_components=2,
    metric="euclidean",
    random_state=SEED
)
X2 = umap_2d.fit_transform(Xp)

pdf["umap_x"] = X2[:, 0]
pdf["umap_y"] = X2[:, 1]

# -----------------------------
# 6) FILE NAME ONLY FOR HOVER
# -----------------------------
pdf["file_name"] = pdf["path"].apply(lambda x: os.path.basename(str(x)))

# -----------------------------
# 7) STRATIFIED SAMPLE FOR PLOT (ENSURES ALL CLUSTERS SHOW)
# -----------------------------
plot_pdf = (pdf.groupby("cluster", group_keys=False)
              .apply(lambda g: g.sample(n=min(len(g), PLOT_PER_CLUSTER), random_state=SEED))
              .reset_index(drop=True))

# Make cluster categorical for distinct colors
plot_pdf["cluster_str"] = plot_pdf["cluster"].astype(str)

# -----------------------------
# 8) INTERACTIVE PLOTLY SCATTER
# -----------------------------
fig = px.scatter(
    plot_pdf,
    x="umap_x",
    y="umap_y",
    color="cluster_str",
    height=800,
    title=f"Document Map (KMeans K={K} + PCA {PCA_K}D + UMAP 2D)"
)

# Hover shows ONLY file name
fig.update_traces(
    customdata=plot_pdf[["file_name"]],
    hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
    marker=dict(size=6, opacity=0.85)
)

fig.update_layout(template="plotly_dark")
fig.show()

# -----------------------------
# 9) OPTIONAL: SAVE CLUSTER ASSIGNMENTS (DOES NOT TOUCH ORIGINAL TABLE)
# -----------------------------
if SAVE_RESULTS:
    out_pdf = pdf[["path", "is_scanned", "cluster"]].copy()
    out_sdf = spark.createDataFrame(out_pdf)
    (out_sdf.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(OUT_TABLE))
    print("\nSaved clusters to:", OUT_TABLE)


In [0]:
out_path = "/dbfs/FileStore/kmeans_k7_umap2d.html"
fig.write_html(out_path)
print("Open in browser at: /files/kmeans_k7_umap2d.html")

In [0]:
import umap
import plotly.express as px

# -----------------------------
# 1) Compute 3D UMAP coordinates
# -----------------------------
umap_3d = umap.UMAP(
    n_neighbors=40,
    min_dist=0.25,
    n_components=3,
    metric="euclidean",
    random_state=42
)

X3 = umap_3d.fit_transform(Xp)

pdf["umap_x"] = X3[:, 0]
pdf["umap_y"] = X3[:, 1]
pdf["umap_z"] = X3[:, 2]

# -----------------------------
# 2) Plot (use a sample so Databricks doesn't drop output)
# -----------------------------
plot_pdf = pdf.sample(n=12000, random_state=42)
plot_pdf["cluster_str"] = plot_pdf["cluster"].astype(str)

fig3d = px.scatter_3d(
    plot_pdf,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="cluster_str",
    height=850,
    title="3D Document Clusters (KMeans + 3D UMAP)"
)

# Hover shows ONLY file name
fig3d.update_traces(
    customdata=plot_pdf[["file_name"]],
    hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
    marker=dict(size=3, opacity=0.85)
)

# -----------------------------
# 3) Save to HTML (best way to view)
# -----------------------------
out_path = "/dbfs/FileStore/kmeans_k7_umap3d.html"
fig3d.write_html(out_path)

print("Saved:", out_path)
print("Open in browser at: /files/kmeans_k7_umap3d.html")
